# Lähde-ekstraktio: {Author Year}

DOI: `{doi}`  
PDF: `data/raw/{path/to/pdf}`

Tämä notebook tekee viisi vaihetta:

1. **Lukee PDF:n** (avaa Claude Code -sivupaneelista)
2. **LLM-raakaekstraktio** Pydantic-skeemalla `RawPair`
3. **Rikastaa** SMILES:t PubChem:istä (RDKit kanonisoi)
4. **Soveltaa** kaksiosaista luokitusta:
   - `classify_gfa_dsc` → `gfa_class_dsc` + `gfa_label_confidence` (Baird et al. 2010)
   - `classify_stability_week_bin` + `classify_stability_protocol` → `stability_week_bin`,
     `stability_protocol_match`, `stability_label_confidence`
5. **Validoi** mole/weight-summat, SMILES, säilytysolot

## Periaate: LLM ei päätä luokituksia

Älä **koskaan** editoi `gfa_class_dsc`-, `gfa_label_confidence`-, `stability_week_bin`- tai
`stability_label_confidence`-kenttiä käsin. Jos rivi näyttää väärin luokitellulta, korjaa sen
syöttötiedot (esim. `crystallizes_on_dsc_cooling`, `crystallizes_on_dsc_reheating`,
`induction_time_censored`, `storage_T_C`, `experimental_protocol`) ja anna luokituksen päivittyä
automaattisesti, kun ajat notebookin uudelleen.

GFA (lasinmuodostuskyky) ja stabiilius ovat **eri ilmiöitä** ja siksi eri sarakkeita:
GFA on molekyylin sisäinen DSC-syklin ominaisuus (kiteytyykö jäähdytyksessä /
uudelleenlämmityksessä), kun taas stabiilius riippuu säilytysoloista (T, RH, kesto).

Sama koskee SMILES-merkkijonoja: ne tulevat PubChem:istä RDKit-kanonisoinnin kautta, ei LLM:n
muistista. Jos PubChem ei tunnista nimeä, rivi merkitään `needs_review=True` ja korjataan käsin
`*_raw.json`-tiedostoon.

## Solu 2 — Imports ja konfiguraatio

In [8]:
import json
from pathlib import Path

import pandas as pd

from coamorphous.corpus.schema import build_schema, load_schema_yaml
from coamorphous.extraction.enrich import (
    assign_pair_id,
    raw_pair_to_master_row,
)
from coamorphous.extraction.extraction_schema import RawPair
from coamorphous.extraction.validate import run_all_validations

# --- Säädettävät parametrit per paperi ---------------------------------------
# Korvaa nämä jokaisen lähteen kohdalla.
PDF_PATH = Path("data/raw/knapik_2019.pdf")
SOURCE_DOI = "10.3390/ph12010040"
SOURCE_FIRST_AUTHOR = "knapik"   # esim. "fink"
SOURCE_YEAR = 2019
EXTRACTED_BY = "claude_code"

# Polut, jotka johdetaan automaattisesti yllä olevista parametreista.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_JSON_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}_raw.json"
INTERIM_CSV_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}.csv"
SCHEMA_YAML = PROJECT_ROOT / "configs" / "corpus_schema.yaml"

print(f"PDF: {PDF_PATH}")
print(f"Raw JSON output: {RAW_JSON_PATH}")
print(f"Interim CSV output: {INTERIM_CSV_PATH}")

PDF: data\raw\knapik_2019.pdf
Raw JSON output: c:\Users\okorhone\coamorphous\data\interim\knapik_2019_raw.json
Interim CSV output: c:\Users\okorhone\coamorphous\data\interim\knapik_2019.csv


## Vaihe 1 — LLM-raakaekstraktio (automatisoitu)

Tämä solu kutsuu **Claude Code -CLI:tä** subprocessina, joka:

1. Lukee PDF:n (`PDF_PATH`)
2. Ekstraktoi co-amorfiset parit `RawPair`-skeeman mukaisesti
3. Tallentaa raaka-JSONin polkuun `RAW_JSON_PATH`

Aikaisemmin tämä vaihe vaati promptin manuaalisen kopioinnin Claude Coden sivupaneeliin. Nyt ekstraktio ajetaan kerralla notebookin sisältä — sama LLM, sama prompt, mutta ilman copy-paste-vaihetta.

### Vaatimukset

- **`claude` CLI** asennettu globaalisti:
  ```bash
  npm install -g @anthropic-ai/claude-code
  ```
- **Sisäänkirjautuminen Max-tilauksellasi**. Jos et ole vielä kirjautunut CLI:llä, aja terminaalissa kerran `claude` ja seuraa OAuth-prosessia. CLI ja VS Code -laajennus voivat käyttää eri credentialeja, joten myös CLI vaatii oman kirjautumisen.
- **PDF saatavilla** polussa `PDF_PATH`.

### Käyttö

Aja alla oleva solu kun olet päivittänyt `PDF_PATH` ja muut parametrit Solu 2:ssa. Ekstraktio kestää tyypillisesti 30–90 sekuntia per PDF; pidemmissä artikkeleissa pidempään.

### Käytetty prompt (auditointia varten)

Sama promptirunko ajetaan jokaiselle PDF:lle, vain `PDF_PATH` ja `RAW_JSON_PATH` substituoidaan. Promptin sisältö on koodissa alla; tässä se näytetään vertailukohdaksi:

```
TEHTÄVÄ: Ekstraktoi co-amorfiset parit PDF:stä JSON-tiedostoon.

KRIITTISET SÄÄNNÖT:
1. Älä päättele tai laske kanonisia SMILES:eja, InChIKeyjä, gfa_class:ia,
   label_confidence:ia tai mitään pari-tason deskriptoreita - ne lasketaan
   automaattisesti.
2. Anna ratio_source_quote eksakti lainaus lähteestä (esim. "1:1 mol"
   tai "70:30 w/w").
3. Jos suhteita on annettu massoina, anna molemmat (weight_fraction ja
   mole_fraction laskettuna massoista, KUNNES Tm:t ovat tiedossa).
4. Aseta induction_time_censored TARKASTI:
   - True = kokeilu loppui ilman havaittua kiteytymistä
   - False = kiteytyminen havaittiin annetulla ajanhetkellä
5. Aseta experimental_protocol parhaalla mahdollisella tarkkuudella:
   - 40 °C / 75 % RH -> ich_q1a_accelerated
   - 25 °C / 60 % RH -> ich_q1a_long_term
   - Kuiva (P2O5/silica), <30 vrk -> dry_short_term
   - Tg+15 K -mittaus -> tg_plus_15K
   - Pelkkä DSC-uusinta -> dsc_in_situ
   - T > Tg + 15 K, kineettinen kiteytymiskoe (BDS, isothermal DSC) -> above_tg_kinetic
   - Muu / epäselvä -> non_standard

   Tg-mittaus: ekstraktoi Tg DSC 10 K/min -arvosta jos paperissa on
   saatavilla. Jos vain TMDSC (esim. 0.5 K/min), 5 K/min tai muu
   nopeus on saatavilla, käytä sitä mutta merkitse Tg_heating_rate_K_min
   oikein. 10 K/min on standardiarvo.

   above_tg_kinetic-protokolla: jos paperi tutkii kiteytymistä
   selvästi Tg:n yläpuolella (esim. T = 373 K kun Tg = 323 K), tämä on
   KINETIC-koe (BDS, isothermal DSC), EI säilyvyyskoe. Tällaiset
   mittaukset eivät ennusta käytännön varastointistabiliteettia.
   Aseta experimental_protocol = "above_tg_kinetic". Älä yritä
   mahduttaa sitä ICH-protokolliin.
6. Aseta needs_review=True ja review_reasons-listalle, jos:
   - Lähde on epäjohdonmukainen
   - Sensurointi on epäselvä
   - Protokolla ei sovi mihinkään selvästi
   - Stabiiliuskoetta ei raportoitu lainkaan (vain DSC tai PXRD)
7. source_quote on 1-2 lauseen lainaus lähteestä, joka tukee tämän parin tietoja.

Tallenna lopputulos JSON-listana tiedostoon: {RAW_JSON_PATH}
```

### Manuaalinen vararatkaisu

Jos automaattinen ajo epäonnistuu (esim. CLI ei ole kirjautunut, PDF on liian iso), voit edelleen avata PDF:n VS Codessa rinnakkain ja kopioida promptin Claude Code -sivupaneeliin manuaalisesti.

In [ ]:
import shutil
import subprocess

# --- 1) Tarkista että claude CLI on PATH:issa --------------------------------
# Jos tämä epäonnistuu, asenna globaalisti: npm install -g @anthropic-ai/claude-code
claude_bin = shutil.which("claude")
if claude_bin is None:
    raise RuntimeError(
        "claude CLI:tä ei löydy PATH:ista. Asenna: "
        "npm install -g @anthropic-ai/claude-code"
    )
print(f"Käytetään claude CLI: {claude_bin}")

# --- 2) Muodosta polut promptia varten ---------------------------------------
# Käytetään suhteellisia polkuja projektin juuresta (Claude operoi cwd=PROJECT_ROOT),
# jotta se voi lukea PDF:n ja kirjoittaa JSONin oman --add-dir-laajuutensa sisällä.
pdf_relative = (
    PDF_PATH.relative_to(PROJECT_ROOT) if PDF_PATH.is_absolute() else PDF_PATH
)
raw_json_relative = RAW_JSON_PATH.relative_to(PROJECT_ROOT)

# --- 3) Kirjoita pitkä prompt tiedostoon ja anna agentille lyhyt argv --------
# MIKSI tiedosto eikä CLI-arg tai stdin:
# Windowsilla `claude` on `.CMD`-batch-skripti. cmd.exe tulkitsee CLI-argumentissa
# olevat newline-merkit komentojen erottimiksi, joten pitkän monirivisen promptin
# vain ensimmäinen rivi päätyy claudelle ja agentti vastaa "polut puuttuvat".
# Stdin (`input=prompt`) ei myöskään välity .CMD-wrapperin läpi luotettavasti
# `-p`-tilassa. Ratkaisu: kirjoita prompt tiedostoon ja anna argv:nä YHDEN rivin
# komento "lue tämä tiedosto ja suorita ohjeet". Agentti käyttää Read-toolia ja
# saa polut tiedostosta puhtaana monirivisenä tekstinä.
prompt_body = f"""TEHTÄVÄ: Ekstraktoi co-amorfiset parit PDF:stä JSON-tiedostoon.

INPUT PDF:           {pdf_relative}
OUTPUT JSON:         {raw_json_relative}
SKEEMA:              src/coamorphous/extraction/extraction_schema.py (RawPair)

VAIHEET (suorita JÄRJESTYKSESSÄ, älä kysy lupaa):
1. Lue PDF Read-toolilla (polku: {pdf_relative}).
2. Tunnista taulukoista ja tekstistä KAIKKI co-amorfiset parit (eri suhteet =
   eri rivit, eri säilytysolot = eri rivit).
3. Muodosta jokaisesta parista RawPair-skeeman mukainen dict KRIITTISTEN
   SÄÄNTÖJEN mukaan (alla).
4. Kirjoita Write-toolilla kaikki parit JSON-listana polkuun: {raw_json_relative}
5. Vahvistus: vastaa pelkkä rivi '[OK] Tallennettu N paria tiedostoon: {raw_json_relative}.'

KRIITTISET SÄÄNNÖT (sovella vaiheessa 3):
1. Älä päättele tai laske kanonisia SMILES:eja, InChIKeyjä, gfa_class_dsc:ia,
   gfa_label_confidence:ia, stability_week_bin:iä, stability_protocol_match:ia tai
   mitään pari-tason deskriptoreita - ne lasketaan automaattisesti myöhemmin.
2. Anna ratio_source_quote eksakti lainaus lähteestä (esim. "1:1 mol"
   tai "70:30 w/w").
3. Jos suhteita on annettu massoina, anna molemmat (weight_fraction ja
   mole_fraction laskettuna massoista, KUNNES Tm:t ovat tiedossa).
4. Aseta induction_time_censored TARKASTI:
   - True = kokeilu loppui ilman havaittua kiteytymistä
   - False = kiteytyminen havaittiin annetulla ajanhetkellä
5. Aseta experimental_protocol parhaalla mahdollisella tarkkuudella:
   - 40 °C / 75 % RH -> ich_q1a_accelerated
   - 25 °C / 60 % RH -> ich_q1a_long_term
   - Kuiva (P2O5/silica), <30 vrk -> dry_short_term
   - Tg+15 K -mittaus -> tg_plus_15K
   - Pelkkä DSC-uusinta -> dsc_in_situ
   - T > Tg + 15 K, kineettinen kiteytymiskoe (BDS, isothermal DSC) -> above_tg_kinetic
   - Muu / epäselvä -> non_standard

   Tg-mittaus: ekstraktoi Tg DSC 10 K/min -arvosta jos paperissa on saatavilla.
   Jos vain TMDSC (esim. 0.5 K/min), 5 K/min tai muu nopeus on saatavilla, käytä
   sitä mutta merkitse Tg_heating_rate_K_min oikein. 10 K/min on standardiarvo.

   above_tg_kinetic-protokolla: jos paperi tutkii kiteytymistä selvästi Tg:n
   yläpuolella (esim. T = 373 K kun Tg = 323 K), tämä on KINETIC-koe (BDS,
   isothermal DSC), EI säilyvyyskoe. Tällaiset mittaukset eivät ennusta
   käytännön varastointistabiliteettia. Aseta experimental_protocol =
   "above_tg_kinetic". Älä yritä mahduttaa sitä ICH-protokolliin.
6. DSC-syklin GFA-havainnot (Baird et al. 2010):
   - crystallizes_on_dsc_cooling: True jos jäähdytyssyklissä havaitaan
     kiteytymispiikki, False jos ei, None jos jäähdytyssykliä ei raportoitu.
   - crystallizes_on_dsc_reheating: True jos uudelleenlämmityssyklissä havaitaan
     kiteytymispiikki, False jos ei, None jos uudelleenlämmitystä ei raportoitu.
   - paper_states_gfa_class: 1/2/3 jos artikkeli ilmoittaa luokan suoraan
     (esim. "Class II glass former"), muuten None. Tätä käytetään vain jos
     DSC-sykliä ei raportoida.
7. Aseta needs_review=True ja review_reasons-listalle, jos:
   - Lähde on epäjohdonmukainen (esim. taulukon otsikko vs. tekstin numerot)
   - Sensurointi on epäselvä
   - Protokolla ei sovi mihinkään selvästi
   - Stabiiliuskoetta ei raportoitu lainkaan (vain DSC tai PXRD)
8. source_quote on 1-2 lauseen lainaus lähteestä, joka tukee tämän parin tietoja.

ÄLÄ anna yhteenvetoja, älä kysy tarkennuksia, älä luettele vaihtoehtoja.
Suorita vaiheet 1-5 ja vastaa vahvistuksella."""

# Tallennetaan prompt data/interim/-hakemistoon, jotta se pysyy projektin sisällä
# (Claude voi sen lukea omalla --add-dir-laajuudellaan) ja jotta se on tarvittaessa
# nähtävissä auditointia varten. Yksi promptitiedosto per ekstraktio, suhteellisella
# polulla joka mahtuu yhden rivin sisään.
prompt_file_path = RAW_JSON_PATH.parent / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}_prompt.txt"
prompt_file_path.write_text(prompt_body, encoding="utf-8")
prompt_file_relative = prompt_file_path.relative_to(PROJECT_ROOT)
print(f"Prompt kirjoitettu: {prompt_file_relative} ({len(prompt_body)} merkkiä)")

# Yksirivinen argv-prompt: agentti lukee koko monirivisen ohjeen tiedostosta.
# Tämä on TURVALLINEN — yksi rivi, ei newlineja, ei cmd.exe-sotkua.
argv_prompt = (
    f"Lue Read-toolilla tiedosto '{prompt_file_relative}' ja suorita siellä "
    f"olevat ohjeet TARKASTI. ÄLÄ kysy lisätietoja — ohjeissa on kaikki polut."
)

# System prompt: agentin rooli yhdellä lauseella.
system_prompt = (
    "Olet data-ekstraktioagentti coamorphous-tutkimusprojektissa. "
    "Sinulle annetaan polku tekstitiedostoon, joka sisältää tehtävän kuvauksen. "
    "Lue tiedosto Read-toolilla ja suorita ohjeet eksaktisti. "
    "ÄLÄ kysy lisätietoja, ÄLÄ anna yhteenvetoja. Lopuksi vastaa yhdellä rivillä."
)

# --- 4) Tyhjennä vanha JSON ennen ajoa ---------------------------------------
# Näin näemme luotettavasti onko Claude tällä ajolla luonut tiedoston.
if RAW_JSON_PATH.exists():
    RAW_JSON_PATH.unlink()
    print(f"Poistettu vanha: {RAW_JSON_PATH.name}")

# --- 5) Aja claude subprocessina ---------------------------------------------
# Lippujen merkitys:
#   -p / --print               = non-interactive print mode (tulosta vastaus + poistu)
#   --append-system-prompt     = lisää extraktioagentti-rooli oletus-system-promptiin
#   --allowedTools             = sallitaan vain Read (PDF + prompt-tiedosto) + Write (JSON)
#   --permission-mode bypassPermissions
#                              = -p-tilassa permission-promptit eivät toimi; rajaus
#                                tapahtuu --allowedTools:lla.
cmd = [
    claude_bin,
    "-p",
    "--append-system-prompt", system_prompt,
    "--allowedTools", "Read,Write",
    "--permission-mode", "bypassPermissions",
    argv_prompt,
]

print(f"\nKutsutaan Claude CLI:tä PDF:lle: {pdf_relative}")
print(f"Tulos kirjoitetaan: {raw_json_relative}")
print(f"Argv-prompt: '{argv_prompt}'")
print("Ekstraktio kestää tyypillisesti 30-90 s per PDF...\n")

result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=900,  # 15 min hard cap
)

# --- 6) Näytä Claude'n vastaus debuggausta varten ----------------------------
if result.stdout:
    print("--- Claude vastaus ---")
    print(result.stdout)

# --- 7) Tarkista onnistuminen ------------------------------------------------
if result.returncode != 0:
    print("\n--- Stderr ---")
    print(result.stderr)
    raise RuntimeError(
        f"claude epäonnistui (exit code {result.returncode}). "
        f"Tarkista yllä oleva tulos."
    )

if not RAW_JSON_PATH.exists():
    raise RuntimeError(
        f"Claude ei luonut tiedostoa: {RAW_JSON_PATH}\n"
        f"Mahdollisia syitä: PDF:ää ei löytynyt, CLI ei kirjautunut, "
        f"prompt-tiedostoa ei luettu, tai Claude antoi yhteenvedon eikä "
        f"kutsunut Write-toolia. Tarkista yllä oleva vastaus ja "
        f"prompt-tiedosto: {prompt_file_relative}"
    )

print(f"\n[OK] Tallennettu: {raw_json_relative}")
print("Jatka seuraavaan soluun (Pydantic-validointi).")

In [10]:
with open(RAW_JSON_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

# Pydantic-validointi: kentät, tyypit, enumit, mole_fraction-rajat.
raw_pairs = [RawPair(**d) for d in raw_data]
print(f"Validoitu {len(raw_pairs)} paria.")

# Listaa heti needs_review-merkityt rivit, jotta ne näkyvät ennen rikastusta.
needs_review_now = [(i, p) for i, p in enumerate(raw_pairs) if p.needs_review]
if needs_review_now:
    print(f"\n[!] {len(needs_review_now)} paria merkitty needs_review jo ekstraktiossa:")
    for i, p in needs_review_now:
        print(f"  [{i}] {p.drug_A_name_raw}-{p.drug_B_name_raw}: {p.review_reasons}")

Validoitu 2 paria.

[!] 1 paria merkitty needs_review jo ekstraktiossa:
  [1] ezetimibe-simvastatin: ['Storage at T > Tg (supercooled liquid), well outside ICH Q1A protocols', 'RH not reported for 373 K experiment']


## Vaihe 2 — Rikasta + luokittele

`raw_pair_to_master_row` tekee per pari:

1. Hakee PubChem:istä SMILES + InChIKey + CID molemmille lääkkeille
2. RDKit-kanonisoinnin
3. **GFA-luokituksen** (`classify_gfa_dsc`) DSC-syklin havainnoista:
   - `gfa_class_dsc` (1/2/3 Baird et al. 2010 mukaan)
   - `gfa_label_confidence` (high jos suora DSC-havainto, low jos vain artikkelin maininta)
   - `gfa_dsc_evidence` (dsc_full_cycle / dsc_cooling_only / paper_classification / not_reported)
4. **Stabiiliusluokituksen** säilytysoloista ja induktioajasta:
   - `stability_week_bin` (`<1w`, `1-2w`, ..., `11-12m`, `>=12m`, `unknown`)
   - `stability_protocol_match` (ich_q1a_accelerated / long_term / dry_short_term / ... / non_standard)
   - `stability_label_confidence` (high vain ICH Q1A -protokollille, muille low)
5. Yhdistää lähdemetadatan ja palauttaa kaikki 58 master-CSV:n saraketta

PubChem-haku ei ole instant — varaa muutama sekunti per pari.

In [11]:
source_metadata = {
    "source_doi": SOURCE_DOI,
    "source_first_author": SOURCE_FIRST_AUTHOR,
    "source_year": SOURCE_YEAR,
}

rows = [
    raw_pair_to_master_row(p, source_metadata, EXTRACTED_BY)
    for p in raw_pairs
]
rows = assign_pair_id(rows, SOURCE_FIRST_AUTHOR, SOURCE_YEAR)

df = pd.DataFrame(rows)
print(f"Rikastettu {len(df)} riviä, {len(df.columns)} saraketta.")

# GFA-luokitus (molekyylin sisäinen DSC-ominaisuus, Baird et al. 2010).
print("\nGFA-luokitus (gfa_class_dsc x gfa_label_confidence):")
print(df.groupby(["gfa_class_dsc", "gfa_label_confidence"], dropna=False).size())

# Stabiiliusluokitus (säilytysolojen tulos — eri ilmiö kuin GFA).
print("\nStabiiliuden week_bin (induction_time_days -> diskretisoitu):")
print(df["stability_week_bin"].value_counts(dropna=False))
print("\nStabiilius-protokollat:")
print(df.groupby(
    ["stability_protocol_match", "stability_label_confidence"], dropna=False
).size())

Rikastettu 2 riviä, 58 saraketta.

GFA-luokitus (gfa_class_dsc x gfa_label_confidence):
gfa_class_dsc  gfa_label_confidence
3              high                    2
dtype: int64

Stabiiliuden week_bin (induction_time_days -> diskretisoitu):
stability_week_bin
>=12m    1
<1w      1
Name: count, dtype: int64

Stabiilius-protokollat:
stability_protocol_match  stability_label_confidence
ich_q1a_long_term         high                          1
non_standard              low                           1
dtype: int64


## Vaihe 3 — Validoi

Kahdenlaisia tarkistuksia:

* **Riviä-kohti** (`run_all_validations`): mole/weight-summat, SMILES-validius, säilytysolojen yhteensopivuus protokollan kanssa.
* **Skeematason** (Pandera, `build_schema`): tyypit, enumit, vaaditut sarakkeet, sarakejärjestys.

Jos jompikumpi epäonnistuu, korjaa lähdön JSON ja aja Vaihe 2 uudelleen.

In [12]:
validation_errors = []
for idx, row in df.iterrows():
    errors = run_all_validations(row.to_dict())
    if errors:
        validation_errors.append((idx, row["pair_id"], errors))

if validation_errors:
    print(f"[!] {len(validation_errors)} riviä ei läpäissyt riviä-kohti -validointia:")
    for idx, pid, errs in validation_errors:
        print(f"  [{idx}] {pid}:")
        for e in errs:
            print(f"    - {e}")
else:
    print("[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.")

# Pandera-skeemavalidointi yhdistetylle DataFramelle.
schema = build_schema(SCHEMA_YAML)
try:
    schema.validate(df, lazy=True)
    print("[OK] Pandera-skeema validoitu.")
except Exception as e:
    print(f"[!] Pandera-virhe:\n{e}")

[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.
[OK] Pandera-skeema validoitu.


## Vaihe 4 — Tarkista needs_review / low-confidence -rivit

Listataan rivit, jotka vaativat ihmissilmää: joko luokitin antoi `gfa_label_confidence='low'`
(GFA arvattu artikkelin maininnasta DSC-syklin sijaan) tai `stability_label_confidence='low'`
(säilytysprotokolla ei ole ICH Q1A -standardin mukainen). Lisäksi `notes`-kentässä `[review]`-
tagi viittaa LLM:n ekstraktiossa nostettuun epäselvyyteen.

In [13]:
# Master-CSV ei sisällä needs_review-saraketta (se on RawPair-tason kenttä),
# joten käytämme molempia label_confidence-sarakkeita ja notes-kentän [review]-tagia
# proxynä. Rivi nostetaan tarkistettavaksi jos JOMPIKUMPI luokitus on low — GFA-luokituksen
# epävarmuus ja stabiiliuden epävarmuus ovat eri ilmiöitä, mutta kumpi tahansa riittää
# perusteeksi käsintarkistukselle.
review_mask = (
    df["gfa_label_confidence"].eq("low")
    | df["stability_label_confidence"].eq("low")
    | df["notes"].fillna("").str.contains(r"\[review\]", regex=True)
)
review_df = df[review_mask]

if len(review_df) > 0:
    print(f"Tarkista {len(review_df)} riviä manuaalisesti:")
    display_cols = [
        "pair_id", "drug_A_name", "drug_B_name",
        "gfa_class_dsc", "gfa_label_confidence", "gfa_dsc_evidence",
        "stability_week_bin", "stability_protocol_match", "stability_label_confidence",
        "experimental_protocol", "notes",
    ]
    print(review_df[display_cols].to_string())
else:
    print("[OK] Ei review-merkittyjä eikä low-confidence-rivejä.")

Tarkista 1 riviä manuaalisesti:
         pair_id drug_A_name  drug_B_name  gfa_class_dsc gfa_label_confidence         gfa_dsc_evidence stability_week_bin stability_protocol_match stability_label_confidence experimental_protocol                                                                                                                                                                                                                                                                                                notes
1  knapik2019_02   ezetimibe  simvastatin              3                 high  dsc_cycle_full_reported                <1w             non_standard                        low          non_standard  Stored at elevated temperature (T = 373 K, ~50 K above Tg), supercooled liquid state. Recrystallization onset at 210 min (3.5 h = 0.146 d). Total experiment duration 30 h. | [review] Storage at T > Tg (supercooled liquid), well outside ICH Q1A protocols; RH not reported for 373 K e

## Vaihe 5 — Tallenna interim-CSV

In [14]:
df.to_csv(INTERIM_CSV_PATH, index=False)
print(f"[OK] Tallennettu: {INTERIM_CSV_PATH} ({len(df)} riviä)")
print("\nSEURAAVAT TOIMET:")
print("1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.")
print(f"2. Editoi tarvittaessa {RAW_JSON_PATH.name}, aja Vaihe 2-5 uudelleen.")
print("3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.")

[OK] Tallennettu: c:\Users\okorhone\coamorphous\data\interim\knapik_2019.csv (2 riviä)

SEURAAVAT TOIMET:
1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.
2. Editoi tarvittaessa knapik_2019_raw.json, aja Vaihe 2-5 uudelleen.
3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.
